<a href="https://colab.research.google.com/github/hemanginivyas3/itaassignment/blob/main/ITA%20Assignment%20Project%202.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task
Analyze the 'NFHS5_India_Disease_Risk_Dataset.csv' file to:
1. Calculate descriptive statistics for disease-related columns, identify top/bottom states for each disease (rural), and determine the highest national rural-urban disparity.
2. Conduct Pearson correlation analysis between specified disease pairs (diabetes women rural vs hypertension women rural, diabetes men rural vs hypertension men rural, anaemia children rural vs anaemia all women rural) and interpret for rural health policy.
3. Create and rank a composite disease burden score for each state based on six specified rural disease columns, classifying states into HIGH, MEDIUM, and LOW risk categories.
4. Assess ML readiness by identifying 5 strong predictive features for disease risk, determining the disease with the worst class imbalance and recommending SMOTE ratio, comparing Random Forest and Logistic Regression suitability, and recommending AUC-ROC and Recall targets.
5. Formulate 5 key findings for a research report, develop a 3-sentence policy recommendation for ASHA workers, and identify 2 states requiring urgent intervention with explanations.
Present all findings with clean section headers, relevant tables, and specific numbers, suitable for a 5-page academic research report on ML-based disease risk prediction for rural India.

## Load Dataset

### Subtask:
Load the 'NFHS5_India_Disease_Risk_Dataset.csv' file into a pandas DataFrame for analysis.


**Reasoning**:
To load the dataset, I need to import the pandas library, read the specified CSV file into a DataFrame, and then display the head of the DataFrame to confirm it loaded correctly.



In [1]:
import pandas as pd

df = pd.read_csv('/content/NFHS5_India_Disease_Risk_Dataset.csv')

print(df.head())

               State            Survey              Source  \
0              India  NFHS-5 (2019-21)  MoHFW India / IIPS   
1  Arunachal Pradesh  NFHS-5 (2019-21)  MoHFW India / IIPS   
2       Chhattisgarh  NFHS-5 (2019-21)  MoHFW India / IIPS   
3            Haryana  NFHS-5 (2019-21)  MoHFW India / IIPS   
4          Jharkhand  NFHS-5 (2019-21)  MoHFW India / IIPS   

   anaemia_children_under5_urban_%  anaemia_children_under5_rural_%  \
0                             54.1                             58.7   
1                             37.0                             41.6   
2                             57.1                             62.5   
3                             57.5                             62.1   
4                             61.6                             67.0   

   anaemia_children_under5_total_%  anaemia_nonpreg_women_urban_%  \
0                             57.2                           45.7   
1                             40.8                           2

## SECTION 1 - Descriptive Statistics

### Subtask:
Calculate descriptive statistics for all disease-related columns, identify top 3 and bottom 3 states for each rural disease, and determine the disease with the highest national rural-urban disparity.


**Reasoning**:
The first step is to prepare the data by filtering out the national average row for 'India' and then identifying all disease prevalence columns (those ending with '_%') to calculate their descriptive statistics. This will give a foundational understanding of the data distribution at the state level.



In [2]:
import numpy as np

# 1. Create a new DataFrame by filtering out the row where the 'State' column is 'India'
df_states = df[df['State'] != 'India'].copy()

# 2. Identify all columns in this new DataFrame that end with '_%'
disease_prevalence_cols = [col for col in df_states.columns if col.endswith('_%')]

# 3. Calculate the descriptive statistics for these identified disease prevalence columns
#    and format them clearly in a table.
descriptive_stats = df_states[disease_prevalence_cols].describe().loc[['mean', '50%', 'std', 'min', 'max']]
descriptive_stats.rename(index={'50%': 'median'}, inplace=True)

print("### Descriptive Statistics for Disease Prevalence Columns (State-Level):")
print(descriptive_stats.to_markdown(numalign="left", stralign="left"))

### Descriptive Statistics for Disease Prevalence Columns (State-Level):
|        | anaemia_children_under5_urban_%   | anaemia_children_under5_rural_%   | anaemia_children_under5_total_%   | anaemia_nonpreg_women_urban_%   | anaemia_nonpreg_women_rural_%   | anaemia_nonpreg_women_total_%   | anaemia_preg_women_urban_%   | anaemia_preg_women_rural_%   | anaemia_preg_women_total_%   | anaemia_all_women_urban_%   | anaemia_all_women_rural_%   | anaemia_all_women_total_%   | anaemia_men_15to49_urban_%   | anaemia_men_15to49_rural_%   | anaemia_men_15to49_total_%   | diabetes_women_urban_%   | diabetes_women_rural_%   | diabetes_women_total_%   | diabetes_men_urban_%   | diabetes_men_rural_%   | diabetes_men_total_%   | hypertension_women_urban_%   | hypertension_women_rural_%   | hypertension_women_total_%   | hypertension_men_urban_%   | hypertension_men_rural_%   | hypertension_men_total_%   |
|:-------|:----------------------------------|:----------------------------------|:-----------

**Reasoning**:
Now that descriptive statistics are calculated, the next step is to identify all rural disease prevalence columns, then iterate through each to find and display the top 3 and bottom 3 states based on their prevalence values. This directly addresses parts 4 and 5 of the subtask.



In [3]:
import re

# 4. Identify all columns in the new DataFrame that end specifically with '_rural_%'
rural_disease_cols = [col for col in df_states.columns if col.endswith('_rural_%')]

print("### Top 3 and Bottom 3 States for Each Rural Disease:\n")

# 5. For each of these '_rural_%' columns:
for col in rural_disease_cols:
    print(f"#### Disease: {col.replace('_rural_%', '').replace('_', ' ').title()}\n")

    # a. Find and list the top 3 states with the highest prevalence values.
    top_3_states = df_states[['State', col]].sort_values(by=col, ascending=False).head(3)
    print("##### Top 3 States (Highest Prevalence):\n")
    print(top_3_states.to_markdown(numalign="left", stralign="left"))
    print("\n")

    # b. Find and list the bottom 3 states with the lowest prevalence values.
    bottom_3_states = df_states[['State', col]].sort_values(by=col, ascending=True).head(3)
    print("##### Bottom 3 States (Lowest Prevalence):\n")
    print(bottom_3_states.to_markdown(numalign="left", stralign="left"))
    print("\n---\n")

# 6. For each disease category that has both a rural and an urban counterpart:
#    a. Calculate the absolute difference between the rural percentage and the urban percentage for each state in that category.

urban_rural_pairs = []
# Extract base names for diseases (e.g., 'anaemia_children_under5')
# Find urban columns and check for a corresponding rural column
for col in df_states.columns:
    if col.endswith('_urban_%'):
        base_name = col.replace('_urban_%', '')
        rural_col = base_name + '_rural_%'
        if rural_col in df_states.columns:
            urban_rural_pairs.append((col, rural_col, base_name))

max_disparity = 0
max_disparity_disease = ''
max_disparity_state = ''

for urban_col, rural_col, base_name in urban_rural_pairs:
    # Calculate absolute difference for each state
    df_states[f'{base_name}_disparity'] = abs(df_states[urban_col] - df_states[rural_col])

    # Find the state with the maximum disparity for this specific disease
    current_max_disparity_row = df_states.loc[df_states[f'{base_name}_disparity'].idxmax()]
    current_max_disparity_value = current_max_disparity_row[f'{base_name}_disparity']

    # Update overall maximum disparity if current one is greater
    if current_max_disparity_value > max_disparity:
        max_disparity = current_max_disparity_value
        max_disparity_disease = base_name.replace('_', ' ').title()
        max_disparity_state = current_max_disparity_row['State']

print("### Highest National Rural-Urban Disparity:\n")
print("##### Identified Disease Category, State, and Maximum Disparity Value:\n")
print(f"Disease Category: {max_disparity_disease}")
print(f"State: {max_disparity_state}")
print(f"Maximum Disparity: {max_disparity:.2f} percentage points")

### Top 3 and Bottom 3 States for Each Rural Disease:

#### Disease: Anaemia Children Under5

##### Top 3 States (Highest Prevalence):

|    | State        | anaemia_children_under5_rural_%   |
|:---|:-------------|:----------------------------------|
| 4  | Jharkhand    | 67                                |
| 6  | Odisha       | 65                                |
| 2  | Chhattisgarh | 62.5                              |


##### Bottom 3 States (Lowest Prevalence):

|    | State             | anaemia_children_under5_rural_%   |
|:---|:------------------|:----------------------------------|
| 11 | Uttarakhand       | 41                                |
| 1  | Arunachal Pradesh | 41.6                              |
| 10 | Uttar Pradesh     | 50.7                              |

---

#### Disease: Anaemia Nonpreg Women

##### Top 3 States (Highest Prevalence):

|    | State     | anaemia_nonpreg_women_rural_%   |
|:---|:----------|:--------------------------------|
| 6  | Odisha    | 62.

## SECTION 2 - Correlation Analysis

### Subtask:
Calculate Pearson correlation between diabetes women rural vs hypertension women rural, diabetes men rural vs hypertension men rural, and anaemia children rural vs anaemia all women rural. Interpret the meaning of each correlation for rural health policy.


**Reasoning**:
The next logical step is to calculate the Pearson correlation coefficients for the specified disease pairs using the `df_states` DataFrame and then interpret these results for rural health policy, as outlined in the instructions.



In [4]:
print("### Pearson Correlation Analysis for Rural Health Policy\n")

# Pair 1: Diabetes women rural vs Hypertension women rural
corr_dw_hw = df_states['diabetes_women_rural_%'].corr(df_states['hypertension_women_rural_%'])
print(f"#### 1. Correlation between Diabetes (Women, Rural) and Hypertension (Women, Rural):\n")
print(f"Pearson Correlation Coefficient: {corr_dw_hw:.2f}\n")
print("##### Interpretation for Rural Health Policy:")
print("A positive correlation suggests that states with higher diabetes prevalence among rural women also tend to have higher hypertension prevalence among rural women. This could indicate shared risk factors (e.g., lifestyle, diet, obesity) or common healthcare access issues. Policy implications include integrated screening programs for both conditions, and health education campaigns targeting risk factors common to both diseases for rural women.\n")

# Pair 2: Diabetes men rural vs Hypertension men rural
corr_dm_hm = df_states['diabetes_men_rural_%'].corr(df_states['hypertension_men_rural_%'])
print(f"#### 2. Correlation between Diabetes (Men, Rural) and Hypertension (Men, Rural):\n")
print(f"Pearson Correlation Coefficient: {corr_dm_hm:.2f}\n")
print("##### Interpretation for Rural Health Policy:")
print("Similar to women, a positive correlation here suggests a link between diabetes and hypertension in rural men. This points towards the need for comprehensive health programs that address metabolic syndrome components. Policies should consider male-specific health-seeking behaviors and occupational health risks that might contribute to both conditions in rural settings.\n")

# Pair 3: Anaemia children rural vs Anaemia all women rural
corr_ac_aw = df_states['anaemia_children_under5_rural_%'].corr(df_states['anaemia_all_women_rural_%'])
print(f"#### 3. Correlation between Anaemia (Children, Rural) and Anaemia (All Women, Rural):\n")
print(f"Pearson Correlation Coefficient: {corr_ac_aw:.2f}\n")
print("##### Interpretation for Rural Health Policy:")
print("A strong positive correlation between anaemia in rural children and rural women is expected, as maternal anaemia is a significant risk factor for childhood anaemia, and both often stem from similar nutritional deficiencies or poor access to iron and folic acid supplementation. This highlights the importance of integrated nutritional programs focusing on pregnant women, lactating mothers, and young children. Interventions like universal iron supplementation and improved maternal and child health services in rural areas could address this linked burden effectively.\n")

### Pearson Correlation Analysis for Rural Health Policy

#### 1. Correlation between Diabetes (Women, Rural) and Hypertension (Women, Rural):

Pearson Correlation Coefficient: 0.56

##### Interpretation for Rural Health Policy:
A positive correlation suggests that states with higher diabetes prevalence among rural women also tend to have higher hypertension prevalence among rural women. This could indicate shared risk factors (e.g., lifestyle, diet, obesity) or common healthcare access issues. Policy implications include integrated screening programs for both conditions, and health education campaigns targeting risk factors common to both diseases for rural women.

#### 2. Correlation between Diabetes (Men, Rural) and Hypertension (Men, Rural):

Pearson Correlation Coefficient: 0.48

##### Interpretation for Rural Health Policy:
Similar to women, a positive correlation here suggests a link between diabetes and hypertension in rural men. This points towards the need for comprehensive h

## SECTION 3 - Composite Disease Burden Score

### Subtask:
Create a composite disease burden score for each state, rank them, and classify them into risk categories.


**Reasoning**:
The first step in creating the composite burden score is to define the relevant columns, calculate their average for each state, and then rank the states based on this score. This directly addresses the first three instructions of the subtask.



In [5]:
print("### Composite Disease Burden Score Calculation\n")

# 1. Define a list of the six rural disease columns
disease_columns = [
    'anaemia_children_under5_rural_%',
    'anaemia_all_women_rural_%',
    'diabetes_women_rural_%',
    'diabetes_men_rural_%',
    'hypertension_women_rural_%',
    'hypertension_men_rural_%'
]

# 2. Calculate the average of these six columns for each state
df_states['composite_burden_score'] = df_states[disease_columns].mean(axis=1)

# 3. Rank the states from highest to lowest composite burden score
df_states_ranked = df_states.sort_values(by='composite_burden_score', ascending=False).reset_index(drop=True)
df_states_ranked['rank'] = df_states_ranked.index + 1

print("##### States Ranked by Composite Disease Burden Score (Top 5):\n")
print(df_states_ranked[['State', 'composite_burden_score', 'rank']].head().to_markdown(index=False, numalign="left", stralign="left"))

### Composite Disease Burden Score Calculation

##### States Ranked by Composite Disease Burden Score (Top 5):

| State      | composite_burden_score   | rank   |
|:-----------|:-------------------------|:-------|
| Odisha     | 35.8                     | 1      |
| Punjab     | 35.75                    | 2      |
| Tamil Nadu | 34.4667                  | 3      |
| Chandigarh | 34.4                     | 4      |
| Puducherry | 34.25                    | 5      |


**Reasoning**:
Now that the composite burden score and initial ranking have been established, the next step is to calculate the 33rd and 66th percentiles of this score. These percentiles will then be used to classify each state into 'HIGH RISK', 'MEDIUM RISK', or 'LOW RISK' categories, and finally present the results in a formatted table.



In [6]:
print("### State Classification by Disease Risk Category\n")

# 4. Calculate the 33rd and 66th percentiles of the composite_burden_score
percentile_33 = df_states_ranked['composite_burden_score'].quantile(0.33)
percentile_66 = df_states_ranked['composite_burden_score'].quantile(0.66)

print(f"33rd Percentile (Low/Medium Threshold): {percentile_33:.2f}\n")
print(f"66th Percentile (Medium/High Threshold): {percentile_66:.2f}\n")

# 5. Classify each state into 'HIGH RISK', 'MEDIUM RISK', and 'LOW RISK' categories
def classify_risk(score):
    if score >= percentile_66:
        return 'HIGH RISK'
    elif score >= percentile_33:
        return 'MEDIUM RISK'
    else:
        return 'LOW RISK'

df_states_ranked['risk_category'] = df_states_ranked['composite_burden_score'].apply(classify_risk)

# 6. Present the state, its composite burden score, rank, and risk category in a clean, formatted table, sorted by rank.
print("##### States Classified by Risk Category (Sorted by Rank):\n")
print(df_states_ranked[['State', 'composite_burden_score', 'rank', 'risk_category']].to_markdown(index=False, numalign="left", stralign="left"))

### State Classification by Disease Risk Category

33rd Percentile (Low/Medium Threshold): 31.11

66th Percentile (Medium/High Threshold): 33.99

##### States Classified by Risk Category (Sorted by Rank):

| State             | composite_burden_score   | rank   | risk_category   |
|:------------------|:-------------------------|:-------|:----------------|
| Odisha            | 35.8                     | 1      | HIGH RISK       |
| Punjab            | 35.75                    | 2      | HIGH RISK       |
| Tamil Nadu        | 34.4667                  | 3      | HIGH RISK       |
| Chandigarh        | 34.4                     | 4      | HIGH RISK       |
| Puducherry        | 34.25                    | 5      | HIGH RISK       |
| Jharkhand         | 33.6333                  | 6      | MEDIUM RISK     |
| Haryana           | 33.0833                  | 7      | MEDIUM RISK     |
| NCT Delhi         | 32.4333                  | 8      | MEDIUM RISK     |
| Chhattisgarh      | 32.2333     

## SECTION 4 - ML Readiness Assessment

### Subtask:
Identify 5 strong predictive features for disease risk, determine the disease with the worst class imbalance and recommend SMOTE ratio, compare Random Forest and Logistic Regression suitability, and recommend AUC-ROC and Recall targets.


**Reasoning**:
The first step is to calculate the Pearson correlation coefficient between each rural disease prevalence column and the `composite_burden_score` to identify potential predictive features. This involves iterating through the `rural_disease_cols` and computing the correlation with the `composite_burden_score`, storing the results for further analysis.



In [7]:
print("### Identifying Strong Predictive Features for Disease Risk\n")

# 1. Calculate the Pearson correlation coefficient between each rural disease prevalence column (features)
# and the `composite_burden_score` (target).

# Ensure 'composite_burden_score' is present in df_states, if not, recalculate it
# (it was created in df_states_ranked, so we merge it back or re-calculate it on df_states)
# Re-calculating to be safe, as df_states might not have it if earlier steps were not fully run in this session.

# Define a list of the six rural disease columns for composite score, as used previously
composite_disease_cols_for_score = [
    'anaemia_children_under5_rural_%',
    'anaemia_all_women_rural_%',
    'diabetes_women_rural_%',
    'diabetes_men_rural_%',
    'hypertension_women_rural_%',
    'hypertension_men_rural_%'
]

# Calculate the average of these six columns for each state to ensure composite_burden_score exists in df_states
df_states['composite_burden_score'] = df_states[composite_disease_cols_for_score].mean(axis=1)

correlations = {}
for col in rural_disease_cols:
    if col in df_states.columns and 'composite_burden_score' in df_states.columns:
        correlation = df_states[col].corr(df_states['composite_burden_score'])
        correlations[col] = correlation

# Convert to a pandas Series for easy sorting
correlation_series = pd.Series(correlations)

# 2. Sort these correlations by their absolute values in descending order.
sorted_correlations = correlation_series.abs().sort_values(ascending=False)

# 3. Select the top 5 features with the highest absolute correlation values.
top_5_features = sorted_correlations.head(5)

print("#### Top 5 Predictive Features (Rural Disease Columns) for Composite Burden Score:\n")
print(top_5_features.to_markdown(numalign="left", stralign="left"))

print("\n##### Statistical Justification:")
print("These features exhibit the strongest absolute Pearson correlation coefficients with the composite disease burden score, indicating their significant linear relationship and potential predictive power. A higher absolute correlation implies that changes in these disease prevalences are strongly associated with changes in the overall disease burden.")

### Identifying Strong Predictive Features for Disease Risk

#### Top 5 Predictive Features (Rural Disease Columns) for Composite Burden Score:

|                                 | 0        |
|:--------------------------------|:---------|
| anaemia_preg_women_rural_%      | 0.782816 |
| anaemia_children_under5_rural_% | 0.764185 |
| diabetes_women_rural_%          | 0.722683 |
| diabetes_men_rural_%            | 0.636036 |
| anaemia_all_women_rural_%       | 0.594118 |

##### Statistical Justification:
These features exhibit the strongest absolute Pearson correlation coefficients with the composite disease burden score, indicating their significant linear relationship and potential predictive power. A higher absolute correlation implies that changes in these disease prevalences are strongly associated with changes in the overall disease burden.


**Reasoning**:
Now that the top predictive features are identified, the next step is to analyze class imbalance for each rural disease prevalence column by creating binary classification columns based on the median, counting class occurrences, and calculating imbalance ratios to identify the worst case and recommend a SMOTE ratio.



In [8]:
print("### Class Imbalance Analysis and SMOTE Recommendation\n")

class_imbalance_data = []

# 4. For each rural disease prevalence column:
for col in rural_disease_cols:
    # a. Calculate the median value.
    median_val = df_states[col].median()

    # b. Create a binary classification column where values above the median are class 1 (e.g., 'High Risk')
    # and values at or below the median are class 0 (e.g., 'Low Risk').
    df_states[f'{col}_class'] = (df_states[col] > median_val).astype(int)

    # c. Count the number of occurrences of class 0 and class 1.
    class_counts = df_states[f'{col}_class'].value_counts().sort_index()

    # d. Calculate the class imbalance ratio (number of majority class instances / number of minority class instances).
    # Handle division by zero appropriately.
    class_0_count = class_counts.get(0, 0)
    class_1_count = class_counts.get(1, 0)

    imbalance_ratio = np.nan # Initialize with NaN
    majority_class_count = 0
    minority_class_count = 0

    if class_0_count == 0 and class_1_count == 0:
        imbalance_ratio = 1.0 # No data, consider balanced
    elif class_0_count == 0:
        imbalance_ratio = np.inf # Only class 1, infinite imbalance
        majority_class_count = class_1_count
        minority_class_count = 0
    elif class_1_count == 0:
        imbalance_ratio = np.inf # Only class 0, infinite imbalance
        majority_class_count = class_0_count
        minority_class_count = 0
    else:
        majority_class_count = max(class_0_count, class_1_count)
        minority_class_count = min(class_0_count, class_1_count)
        imbalance_ratio = majority_class_count / minority_class_count

    class_imbalance_data.append({
        'Disease': col,
        'Median': median_val,
        'Class 0 Count': class_0_count,
        'Class 1 Count': class_1_count,
        'Imbalance Ratio': imbalance_ratio
    })

imbalance_df = pd.DataFrame(class_imbalance_data)
imbalance_df = imbalance_df.sort_values(by='Imbalance Ratio', ascending=False).reset_index(drop=True)

print("#### Class Imbalance Ratios for Rural Diseases:\n")
print(imbalance_df.to_markdown(index=False, numalign="left", stralign="left"))

# 5. Identify the rural disease column with the highest class imbalance ratio.
worst_imbalance_disease = imbalance_df.iloc[0]

print(f"\n#### Disease with the Worst Class Imbalance:\n")
print(f"Disease: {worst_imbalance_disease['Disease'].replace('_rural_%', '').replace('_', ' ').title()}")
print(f"Imbalance Ratio: {worst_imbalance_disease['Imbalance Ratio']:.2f}")

# Recommend an appropriate SMOTE ratio.
# SMOTE aims to bring the minority class count up to a desired level relative to the majority class.
# If the ratio is A:B where A > B, SMOTE generates (A-B) synthetic samples for the minority class.
# For sampling_strategy, a common approach is to sample the minority class to have the same number of samples as the majority class,
# or to a specified ratio relative to the majority class.
# Example: if ratio is 2 (majority:minority = 2:1), to balance to 1:1, we need to generate samples for minority class to match majority.
# SMOTE `sampling_strategy` takes a float: ratio of the number of samples in the minority class over the number of samples in the majority class after resampling.

# Calculate SMOTE target ratio for balancing:
# Target: minority_count_after_smote / majority_count = 1 (or 0.5 if we want to double minority)

# If minority_class_count is 0, SMOTE cannot be applied directly in a meaningful way to create new samples.
if pd.isinf(worst_imbalance_disease['Imbalance Ratio']) or worst_imbalance_disease['Class 0 Count'] == 0 or worst_imbalance_disease['Class 1 Count'] == 0:
    recommended_smote_ratio = "SMOTE not applicable or highly challenging due to one-sided class distribution. Consider alternative strategies like outlier detection or generating synthetic data based on domain knowledge."
else:
    # To balance classes, we want the minority class to reach the majority class count.
    # The sampling_strategy parameter in SMOTE is typically the desired ratio of the number of samples in the minority class
    # over the number of samples in the majority class after resampling. For a balanced dataset, this ratio is 1.
    # So, we need to bring the minority class count up to the majority class count.
    # If original counts are minority_count and majority_count:
    # sampling_strategy = {minority_class_label: majority_count}

    # For simplicity of recommendation (if not a dictionary strategy), if target is 1:1, ratio would be 1.
    # If the user wants to specify the 'ratio of the number of samples in the minority class over
    # the number of samples in the majority class after resampling', and we want balanced classes,
    # that ratio should be 1.
    recommended_smote_ratio = 1.0 # This means minority class will be sampled to be equal to majority class
    print(f"Recommended SMOTE sampling_strategy: {recommended_smote_ratio:.2f} (This aims for a 1:1 ratio between minority and majority classes)")

print("\n##### Reasoning for SMOTE Recommendation:")
if isinstance(recommended_smote_ratio, float):
    print(f"A `sampling_strategy` of {recommended_smote_ratio:.2f} is recommended for {worst_imbalance_disease['Disease'].replace('_rural_%', '').replace('_', ' ').title()}. This strategy would resample the minority class to have an equal number of instances as the majority class, thereby creating a balanced dataset for training. Balancing classes is crucial for preventing a model from becoming biased towards the majority class and improving its ability to correctly identify instances of the minority class.")
else:
    print(recommended_smote_ratio)


### Class Imbalance Analysis and SMOTE Recommendation

#### Class Imbalance Ratios for Rural Diseases:

| Disease                         | Median   | Class 0 Count   | Class 1 Count   | Imbalance Ratio   |
|:--------------------------------|:---------|:----------------|:----------------|:------------------|
| anaemia_children_under5_rural_% | 59.15    | 7               | 7               | 1                 |
| anaemia_nonpreg_women_rural_%   | 51.05    | 7               | 7               | 1                 |
| anaemia_preg_women_rural_%      | 58.55    | 7               | 7               | 1                 |
| anaemia_all_women_rural_%       | 57.9     | 7               | 7               | 1                 |
| anaemia_men_15to49_rural_%      | 30.9     | 7               | 7               | 1                 |
| diabetes_women_rural_%          | 10.2     | 7               | 7               | 1                 |
| diabetes_men_rural_%            | 13.15    | 7               | 7      

AttributeError: module 'pandas' has no attribute 'isinf'

**Reasoning**:
The previous code failed because `pd.isinf` was used instead of `np.isinf`. I will correct this by using `np.isinf` to properly check for infinite values in the imbalance ratio, which is a numpy float.



In [9]:
print("### Class Imbalance Analysis and SMOTE Recommendation\n")

class_imbalance_data = []

# 4. For each rural disease prevalence column:
for col in rural_disease_cols:
    # a. Calculate the median value.
    median_val = df_states[col].median()

    # b. Create a binary classification column where values above the median are class 1 (e.g., 'High Risk')
    # and values at or below the median are class 0 (e.g., 'Low Risk').
    df_states[f'{col}_class'] = (df_states[col] > median_val).astype(int)

    # c. Count the number of occurrences of class 0 and class 1.
    class_counts = df_states[f'{col}_class'].value_counts().sort_index()

    # d. Calculate the class imbalance ratio (number of majority class instances / number of minority class instances).
    # Handle division by zero appropriately.
    class_0_count = class_counts.get(0, 0)
    class_1_count = class_counts.get(1, 0)

    imbalance_ratio = np.nan # Initialize with NaN
    majority_class_count = 0
    minority_class_count = 0

    if class_0_count == 0 and class_1_count == 0:
        imbalance_ratio = 1.0 # No data, consider balanced
    elif class_0_count == 0:
        imbalance_ratio = np.inf # Only class 1, infinite imbalance
        majority_class_count = class_1_count
        minority_class_count = 0
    elif class_1_count == 0:
        imbalance_ratio = np.inf # Only class 0, infinite imbalance
        majority_class_count = class_0_count
        minority_class_count = 0
    else:
        majority_class_count = max(class_0_count, class_1_count)
        minority_class_count = min(class_0_count, class_1_count)
        imbalance_ratio = majority_class_count / minority_class_count

    class_imbalance_data.append({
        'Disease': col,
        'Median': median_val,
        'Class 0 Count': class_0_count,
        'Class 1 Count': class_1_count,
        'Imbalance Ratio': imbalance_ratio
    })

imbalance_df = pd.DataFrame(class_imbalance_data)
imbalance_df = imbalance_df.sort_values(by='Imbalance Ratio', ascending=False).reset_index(drop=True)

print("#### Class Imbalance Ratios for Rural Diseases:\n")
print(imbalance_df.to_markdown(index=False, numalign="left", stralign="left"))

# 5. Identify the rural disease column with the highest class imbalance ratio.
worst_imbalance_disease = imbalance_df.iloc[0]

print(f"\n#### Disease with the Worst Class Imbalance:\n")
print(f"Disease: {worst_imbalance_disease['Disease'].replace('_rural_%', '').replace('_', ' ').title()}")
print(f"Imbalance Ratio: {worst_imbalance_disease['Imbalance Ratio']:.2f}")

# Recommend an appropriate SMOTE ratio.
# SMOTE aims to bring the minority class count up to a desired level relative to the majority class.
# If the ratio is A:B where A > B, SMOTE generates (A-B) synthetic samples for the minority class.
# For sampling_strategy, a common approach is to sample the minority class to have the same number of samples as the majority class,
# or to a specified ratio relative to the majority class.
# Example: if ratio is 2 (majority:minority = 2:1), to balance to 1:1, we need to generate samples for minority class to match majority.
# SMOTE `sampling_strategy` takes a float: ratio of the number of samples in the minority class over the number of samples in the majority class after resampling.

# Calculate SMOTE target ratio for balancing:
# Target: minority_count_after_smote / majority_count = 1 (or 0.5 if we want to double minority)

# If minority_class_count is 0, SMOTE cannot be applied directly in a meaningful way to create new samples.
if np.isinf(worst_imbalance_disease['Imbalance Ratio']) or worst_imbalance_disease['Class 0 Count'] == 0 or worst_imbalance_disease['Class 1 Count'] == 0:
    recommended_smote_ratio = "SMOTE not applicable or highly challenging due to one-sided class distribution. Consider alternative strategies like outlier detection or generating synthetic data based on domain knowledge."
else:
    # To balance classes, we want the minority class to reach the majority class count.
    # The sampling_strategy parameter in SMOTE is typically the desired ratio of the number of samples in the minority class
    # over the number of samples in the majority class after resampling. For a balanced dataset, this ratio is 1.
    # So, we need to bring the minority class count up to the majority class count.
    # If original counts are minority_count and majority_count:
    # sampling_strategy = {minority_class_label: majority_count}

    # For simplicity of recommendation (if not a dictionary strategy), if target is 1:1, ratio would be 1.
    # If the user wants to specify the 'ratio of the number of samples in the minority class over
    # the number of samples in the majority class after resampling', and we want balanced classes,
    # that ratio should be 1.
    recommended_smote_ratio = 1.0 # This means minority class will be sampled to be equal to majority class
    print(f"Recommended SMOTE sampling_strategy: {recommended_smote_ratio:.2f} (This aims for a 1:1 ratio between minority and majority classes)")

print("\n##### Reasoning for SMOTE Recommendation:")
if isinstance(recommended_smote_ratio, float):
    print(f"A `sampling_strategy` of {recommended_smote_ratio:.2f} is recommended for {worst_imbalance_disease['Disease'].replace('_rural_%', '').replace('_', ' ').title()}. This strategy would resample the minority class to have an equal number of instances as the majority class, thereby creating a balanced dataset for training. Balancing classes is crucial for preventing a model from becoming biased towards the majority class and improving its ability to correctly identify instances of the minority class.")
else:
    print(recommended_smote_ratio)


### Class Imbalance Analysis and SMOTE Recommendation

#### Class Imbalance Ratios for Rural Diseases:

| Disease                         | Median   | Class 0 Count   | Class 1 Count   | Imbalance Ratio   |
|:--------------------------------|:---------|:----------------|:----------------|:------------------|
| anaemia_children_under5_rural_% | 59.15    | 7               | 7               | 1                 |
| anaemia_nonpreg_women_rural_%   | 51.05    | 7               | 7               | 1                 |
| anaemia_preg_women_rural_%      | 58.55    | 7               | 7               | 1                 |
| anaemia_all_women_rural_%       | 57.9     | 7               | 7               | 1                 |
| anaemia_men_15to49_rural_%      | 30.9     | 7               | 7               | 1                 |
| diabetes_women_rural_%          | 10.2     | 7               | 7               | 1                 |
| diabetes_men_rural_%            | 13.15    | 7               | 7      

### Comparison of Random Forest and Logistic Regression Suitability

#### General Characteristics and Strengths:

*   **Random Forest (RF)**:
    *   **Ensemble Method**: Builds multiple decision trees and merges their predictions to get a more accurate and stable prediction.
    *   **Non-linear Relationships**: Capable of capturing complex non-linear relationships between features and the target variable.
    *   **Feature Interactions**: Naturally handles interactions between features.
    *   **Robustness**: Less prone to overfitting than single decision trees, especially with a sufficient number of trees.
    *   **Feature Importance**: Provides a robust measure of feature importance, indicating which variables are most influential in predictions.
    *   **Data Scaling**: Does not require feature scaling.

*   **Logistic Regression (LR)**:
    *   **Linear Model**: Models the probability of a binary outcome using a linear combination of features, passed through a sigmoid function.
    *   **Interpretability**: Highly interpretable, as coefficients directly indicate the impact and direction of each feature on the log-odds of the outcome.
    *   **Efficiency**: Computationally efficient and scales well with large datasets.
    *   **Less Overfitting**: Less prone to overfitting on small datasets or when relationships are predominantly linear.
    *   **Baseline Model**: Often used as a strong baseline for classification tasks.

#### Suitability for this Rural India Dataset (14 states):

Given the extremely small dataset size (14 states), the choice of model is critical, prioritizing robustness and interpretability:

*   **Random Forest**: While powerful, Random Forest models can be prone to overfitting on very small datasets, even with ensemble techniques, because each individual tree still has limited data to learn from. The complexity of RF might not be justified for such limited samples, and its predictions might not generalize well to unseen states. However, its ability to capture non-linear relationships could be beneficial if complex interactions exist.

*   **Logistic Regression**: Due to the small dataset size (N=14), Logistic Regression is generally a more suitable choice. Its simplicity makes it less prone to overfitting with limited data. Crucially, its high interpretability is invaluable for policy recommendations. Policymakers and ASHA workers need clear, actionable insights (e.g., 'a 1% increase in X leads to Y change in disease risk'), which are directly provided by LR coefficients. While it assumes linear relationships, it can still provide a robust baseline, and complex relationships might not be reliably learned from so few data points anyway.

#### Recommendation and Justification:

For this rural India dataset, **Logistic Regression is recommended**.

**Justification**:
1.  **Small Dataset Size**: With only 14 states, models like Random Forest might overfit, leading to poor generalization. Logistic Regression's simpler structure makes it more robust in data-scarce environments.
2.  **Interpretability for Policy**: Policy recommendations, especially for frontline health workers like ASHA, require clear, interpretable models. Logistic Regression provides straightforward coefficients that can be translated into direct insights about risk factors, enabling targeted interventions.
3.  **Baseline Performance**: It serves as an excellent baseline model, and if more data becomes available, more complex models like Random Forest can be explored, using LR as a comparative benchmark.

### Evaluation Metrics and Target Recommendations

#### Importance of AUC-ROC and Recall for Public Health:

*   **AUC-ROC (Area Under the Receiver Operating Characteristic Curve)**:
    *   **Importance**: AUC-ROC measures the overall ability of a model to distinguish between positive and negative classes across various classification thresholds. In public health, a high AUC-ROC indicates that the model is generally good at differentiating between states with high disease risk and those with low disease risk. It is particularly useful when the costs of false positives and false negatives are not explicitly known or are balanced, providing a comprehensive measure of model performance independent of threshold.
    *   **Context**: For a broad assessment of a state's disease risk, AUC-ROC gives a good sense of the model's discriminatory power.

*   **Recall (Sensitivity)**:
    *   **Importance**: Recall (also known as sensitivity) measures the proportion of actual positive cases that are correctly identified by the model. In the context of disease risk prediction, a high Recall means that the model is effective at identifying most of the states that genuinely have a high disease burden. For public health, minimizing false negatives (missing high-risk states) is often paramount, as failing to identify and intervene in high-risk areas can lead to significant negative health outcomes.
    *   **Context**: High Recall is crucial when the priority is to ensure that no high-risk states are overlooked, even if it means a slightly higher rate of false positives (identifying some low-risk states as high-risk, leading to unnecessary but perhaps less harmful interventions).

#### Recommended Target Values and Justification:

For disease risk prediction in rural India, considering the critical implications for public health, the following targets are recommended:

*   **Recommended AUC-ROC Target: > 0.80**
    *   **Justification**: An AUC-ROC greater than 0.80 indicates a good to excellent ability of the model to discriminate between high and low-risk states. While aiming for perfect (1.0) is unrealistic, a score above 0.80 suggests a robust model that consistently performs better than random chance (0.5). This provides confidence in the model's general utility for identifying disease risk profiles across states.

*   **Recommended Recall Target: > 0.70 (or 70%) for the 'High Risk' class**
    *   **Justification**: A Recall target of greater than 0.70 means that the model should correctly identify at least 70% of the truly high-risk states. In public health, **false negatives (missing a high-risk state)** are generally more detrimental than false positives (incorrectly labeling a low-risk state as high-risk). Missing a high-risk state could mean delayed or no intervention, leading to worsening health crises. While false positives might lead to some misallocation of resources, the societal cost of missing genuine high-risk areas is often much higher. A 70% recall ensures that a substantial majority of states needing intervention are indeed flagged by the model, allowing public health agencies to prioritize effectively.

## SECTION 5 - Key Findings for Research Report

### Subtask:
Formulate 5 key findings for a research report, develop a 3-sentence policy recommendation for ASHA workers, and identify 2 states requiring urgent intervention with explanations.


### 5 Key Findings for Research Report:

1.  **High Prevalence of Anaemia Across Rural India**: Descriptive statistics reveal that anaemia in children under 5 and all women in rural areas consistently shows high mean prevalence rates (56.91% and 57.01% respectively), indicating a widespread public health challenge requiring urgent attention.
2.  **Significant Rural-Urban Disparity in Hypertension**: 'Hypertension Women' exhibits the highest national rural-urban disparity, with Rajasthan showing a massive 73.1 percentage point difference, highlighting a critical need to understand and address the unique factors influencing health outcomes in rural versus urban settings.
3.  **Strong Interconnection of Anaemia in Women and Children**: A strong positive Pearson correlation (0.85) between anaemia in rural children and rural women underscores the interconnectedness of maternal and child health. This suggests that interventions targeting women's nutritional status could have a substantial positive impact on child health.
4.  **Identified High-Risk States for Composite Disease Burden**: States like Odisha, Punjab, and Tamil Nadu are classified as 'HIGH RISK' based on their composite disease burden scores, signifying that these regions face a significantly elevated cumulative risk across multiple rural diseases.
5.  **Predictive Power of Anaemia-Related Features**: Anaemia prevalence among pregnant women (rural) and children under 5 (rural) emerged as the strongest predictive features (correlations of 0.78 and 0.76 respectively) for the overall composite disease burden score, suggesting these are crucial indicators for early risk assessment in rural populations.

### Policy Recommendation for ASHA Workers (3 Sentences):

ASHA workers are critical in rural healthcare, and their efforts should be strategically focused. Given the strong correlation between anaemia in women and children, ASHA workers should prioritize integrated screening and nutritional counseling for pregnant women, lactating mothers, and children under five. This targeted approach, coupled with consistent follow-up for iron and folic acid supplementation, will effectively address the linked burden of anaemia at the household level and improve overall rural health outcomes.

### States Requiring Urgent Intervention:

Based on the composite disease burden score, two states requiring urgent intervention are **Odisha** and **Punjab**.

1.  **Odisha**:
    *   **Reasoning**: Odisha ranks 1st with the highest `composite_burden_score` of 35.80, placing it squarely in the 'HIGH RISK' category. This indicates a significantly elevated prevalence across the six measured rural diseases (anaemia in children, women, diabetes in men/women, and hypertension in men/women). The sheer breadth and magnitude of disease burden in Odisha necessitate immediate, multi-faceted public health interventions targeting various demographic groups and disease conditions simultaneously to alleviate this severe health crisis.

2.  **Punjab**:
    *   **Reasoning**: Punjab is ranked 2nd, also falling into the 'HIGH RISK' category with a `composite_burden_score` of 35.75. Its burden score is very close to Odisha's, signaling a similarly critical and widespread health challenge. Earlier analysis also showed Punjab having high prevalence in specific rural diseases (e.g., hypertension men rural), contributing to its high overall composite score. Urgent action is required to address the cumulative disease burden, focusing on prevention, early detection, and management of chronic diseases prevalent in its rural population.

### States Requiring Urgent Intervention:

Based on the composite disease burden score, two states requiring urgent intervention are **Odisha** and **Punjab**.

1.  **Odisha**:
    *   **Reasoning**: Odisha ranks 1st with the highest `composite_burden_score` of 35.80, placing it squarely in the 'HIGH RISK' category. This indicates a significantly elevated prevalence across the six measured rural diseases (anaemia in children, women, diabetes in men/women, and hypertension in men/women). The sheer breadth and magnitude of disease burden in Odisha necessitate immediate, multi-faceted public health interventions targeting various demographic groups and disease conditions simultaneously to alleviate this severe health crisis.

2.  **Punjab**:
    *   **Reasoning**: Punjab is ranked 2nd, also falling into the 'HIGH RISK' category with a `composite_burden_score` of 35.75. Its burden score is very close to Odisha's, signaling a similarly critical and widespread health challenge. Earlier analysis also showed Punjab having high prevalence in specific rural diseases (e.g., hypertension men rural), contributing to its high overall composite score. Urgent action is required to address the cumulative disease burden, focusing on prevention, early detection, and management of chronic diseases prevalent in its rural population.

## Final Task

### Subtask:
Consolidate all outputs with clean section headers, tables where relevant, and specific numbers from the dataset, ensuring the analysis is suitable for a 5-page academic research report on ML-based disease risk prediction for rural India.


## Summary:

### Data Analysis Key Findings

*   **High Prevalence of Anaemia**: Anaemia in rural children under 5 and rural women showed high mean prevalence rates of 56.91% and 57.01%, respectively, indicating a widespread public health challenge.
*   **Significant Rural-Urban Disparity**: Hypertension in women exhibited the highest national rural-urban disparity, with Rajasthan showing a 73.1 percentage point difference, highlighting critical differences between urban and rural health outcomes.
*   **Regional Variations**: Jharkhand (67%) and Odisha (62.2%) reported the highest prevalence of anaemia in rural children under 5 and non-pregnant rural women, respectively. Conversely, Uttarakhand showed the lowest prevalence for both (41% for children, 28.6% for non-pregnant women). Punjab had the highest hypertension prevalence in rural men (36.5%), while Rajasthan had the lowest (19.2%).
*   **Strong Disease Interdependencies**: A strong positive Pearson correlation of 0.85 was observed between anaemia in rural children and rural women. Moderate correlations were found between diabetes and hypertension in rural women (0.56) and in rural men (0.48), suggesting shared risk factors.
*   **Composite Disease Burden**: Odisha (35.80) and Punjab (35.75) were identified as the states with the highest composite disease burden scores, placing them in the 'HIGH RISK' category. States with a score greater than or equal to 33.99 were classified as HIGH RISK, those between 31.11 and 33.99 as MEDIUM RISK, and those less than 31.11 as LOW RISK.
*   **Key Predictive Features**: The top 5 features most correlated with the composite disease burden score were anaemia in pregnant women rural (correlation: 0.78), anaemia in children under 5 rural (correlation: 0.76), diabetes in women rural (correlation: 0.72), diabetes in men rural (correlation: 0.64), and anaemia in all women rural (correlation: 0.59).
*   **ML Model Recommendation**: Given the small dataset size (14 states), Logistic Regression was recommended over Random Forest for disease risk prediction due to its higher interpretability and robustness against overfitting in data-scarce environments.
*   **Evaluation Metrics Targets**: For predictive models, an AUC-ROC target of > 0.80 was recommended for overall discriminative ability, and a Recall target of > 0.70 (70%) for the 'High Risk' class was recommended to minimize false negatives and ensure comprehensive identification of states needing intervention.
*   **States for Urgent Intervention**: Odisha and Punjab were identified as requiring urgent intervention due to their consistently high composite disease burden scores and 'HIGH RISK' classification.

### Insights or Next Steps

*   **Integrated Health Programs**: The strong correlations among various diseases (e.g., anaemia in women and children, diabetes and hypertension) underscore the need for integrated health programs that address shared risk factors and offer comprehensive care rather than siloed interventions, especially in rural settings.
*   **Targeted ASHA Worker Interventions**: ASHA workers should prioritize integrated screening, nutritional counseling, and consistent follow-up for iron and folic acid supplementation among pregnant women, lactating mothers, and young children in rural areas. This targeted approach leverages the strong link between maternal and child anaemia, offering a cost-effective strategy for improving overall rural health outcomes.
*   **Data-Driven Policy Prioritization**: The identification of 'HIGH RISK' states like Odisha and Punjab, coupled with specific ML evaluation targets (high Recall), enables data-driven prioritization of resources and interventions. Future efforts should focus on implementing and evaluating targeted health policies in these critical regions, ensuring that predictive models are optimized to avoid missing genuinely high-risk populations.
